In [2]:
!uv pip install -q sympy numpy "antlr4-python3-runtime==4.11.1"

from google.colab import drive
import os, sys, json, csv, signal
from collections import defaultdict
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import json, re
from collections import Counter

PROJECT_DIR    = '/content/drive/MyDrive/third_try'
SC_PATH        = f'{PROJECT_DIR}/baseline__private__samples.jsonl'
QUESTIONS_PATH = f'{PROJECT_DIR}/private.jsonl'      # for [ANS] counts (the ids you submit)
OUT_CSV        = f'{PROJECT_DIR}/aggregate_submission_v2.csv'

import json, csv, os, sys
from collections import Counter

J = Judger(strict_extract=False)

qcount = {}
for line in open(QUESTIONS_PATH):
    q = json.loads(line); qcount[str(q['id'])] = q.get('question','').count('[ANS]')

def get_text(x):
    if isinstance(x, str): return x
    if isinstance(x, dict):
        ss=[v for v in x.values() if isinstance(v,str)]; return max(ss,key=len) if ss else ""
    return str(x)

def find_all_boxes(s):
    out, i = [], 0
    while True:
        j = s.find('\\boxed{', i)
        if j < 0: break
        k = j+7; d = 1
        while k < len(s) and d: d += (s[k]=='{')-(s[k]=='}'); k += 1
        out.append(s[j+7:k-1]); i = k
    return out

def split_top(s):                       # top-level comma split, NON-destructive (keeps raw text)
    out, depth, start = [], 0, 0
    for i, ch in enumerate(s):
        if ch in '([{<': depth += 1
        elif ch in ')]}>': depth -= 1
        elif ch == ',' and depth <= 0:
            out.append(s[start:i].strip()); start = i+1
    out.append(s[start:].strip())
    return [x for x in out if x]

def sub_answers(text):                   # RAW sub-answers from boxes; None if no box (drops fallback)
    seg = text.rsplit('</think>',1)[-1] if '</think>' in text else text
    bx = find_all_boxes(seg) or find_all_boxes(text)
    if not bx: return None
    items = []
    for b in bx: items += split_top(b)
    return items or None

def is_complete(text):
    te = text.rfind('</think>')
    return te >= 0 and '\\boxed{' in text[te+8:]

def nz(v):
    try: return J.norm_ans_str(v)
    except Exception: return v

def vote_slot(raw_vals):                 # vote on normalized key, RETURN a raw string
    groups = {}
    for v in raw_vals: groups.setdefault(nz(v), []).append(v)
    keys = list(groups)
    parent = {k:k for k in keys}
    def find(k):
        while parent[k]!=k: parent[k]=parent[parent[k]]; k=parent[k]
        return k
    for a in range(len(keys)):
        for b in range(a+1, len(keys)):
            if find(keys[a]) != find(keys[b]):
                try: eq = J.is_equal(keys[a], keys[b])
                except Exception: eq = (keys[a]==keys[b])
                if eq: parent[find(keys[a])] = find(keys[b])
    merged = {}
    for k in keys: merged.setdefault(find(k), []).append(k)
    best = max(merged.values(), key=lambda g: sum(len(groups[k]) for k in g))
    raws = [v for k in best for v in groups[k]]
    return Counter(raws).most_common(1)[0][0]     # raw, real string that appeared

def fmt(s):
    s = s.strip()
    return f'({s})' if len(split_top(s)) > 1 and not (s.startswith('(') and s.endswith(')')) else s

final = {}
for line in open(SC_PATH):
    r = json.loads(line); qid = str(r['id']); is_mc = bool(r.get('is_mc'))
    cand = []                                      # (text, raw_sub_answers, complete)
    for t in map(get_text, r['samples']):
        if not t: continue
        sa = sub_answers(t)
        if sa is None: continue                    # no box -> doesn't vote
        cand.append((t, sa, is_complete(t)))
    if not cand:
        final[qid] = next((get_text(s) for s in r['samples'] if get_text(s)), ""); continue

    if is_mc: K = 1
    else:
        K = qcount.get(qid, 0)
        if K <= 0:
            c = Counter(len(sa) for _,sa,_ in cand); K = c.most_common(1)[0][0]
    well = [p for p in cand if len(p[1]) == K]
    if not well:
        c = Counter(len(sa) for _,sa,_ in cand); K = c.most_common(1)[0][0]
        well = [p for p in cand if len(p[1]) == K]

    voted = [vote_slot([sa[i] for _,sa,_ in well]) for i in range(K)]
    combined = voted[0].strip().upper() if is_mc else ", ".join(fmt(s) for s in voted)
    vkey = tuple(nz(v) for v in voted)
    rep = max(well, key=lambda p: (p[2], sum(nz(a)==b for a,b in zip(p[1], vkey))))
    final[qid] = rep[0].rstrip() + "\n\nThe final answer is \\boxed{" + combined + "}"

os.makedirs(os.path.dirname(OUT_CSV) or '.', exist_ok=True)
with open(OUT_CSV, 'w', newline='') as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL); w.writerow(['id','response'])
    for qid in sorted(final, key=lambda s:(0,int(s)) if s.isdigit() else (1,s)):
        w.writerow([qid, final[qid]])
print('wrote', OUT_CSV, '|', len(final), 'questions')

wrote /content/drive/MyDrive/third_try/aggregate_submission_v2.csv | 943 questions
